# Fine-tuning

In [ ]:
!pip install -U torchao peft transformers

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
import pandas as pd
import numpy as np
import os

In [ ]:
class GBDataset(Dataset):
    def __init__(self, articles_df, annotations_df=None):
        self.annotations_df = annotations_df
        self.articles_df = articles_df

        self.dataset = []

        mismatches = 0

        for idx in range(self.articles_df.shape[0]):
            start_labels = torch.zeros(len(self.articles_df["input_ids"][idx]), dtype=torch.uint8)
            end_labels = torch.zeros(len(self.articles_df["input_ids"][idx]), dtype=torch.uint8)
            span_idxs = []
            span_labels = []
            start_token_to_orig = {}
            end_token_to_orig = {}
            pmid = self.articles_df["pmid"][idx]
            if self.annotations_df is not None:
                for row in self.annotations_df[self.annotations_df['pmid'] == pmid].itertuples(name=None):
                    start_token = None
                    end_token = None

                    ann_start_idx = row[2]
                    ann_end_idx = row[3]

                    # if the entity is in the abstract, shift the idx with the number of chars
                    # added at the start of the string for the title
                    if row[4] == 0:
                        ann_start_idx += self.articles_df['added_chars_no'][idx]
                        ann_end_idx += self.articles_df['added_chars_no'][idx]

                    for i, (offset_start, offset_end) in enumerate(self.articles_df["offset_mapping"][idx]):
                        # Skip special tokens like [CLS] or [SEP] which have (0,0) offsets
                        if offset_start == 0 and offset_end == 0:
                            continue


                        # 1. Capture the start: First token that contains char_start
                        if start_token is None and offset_start <= ann_start_idx < offset_end:
                            start_token = i

                        # 2. Capture the end: Token that contains the last character
                        if offset_start <= ann_end_idx < offset_end:
                            end_token = i


                    if start_token is not None and end_token is not None:
                        start_labels[start_token] = 1
                        end_labels[end_token] = 1
                        span_idxs.append([start_token, end_token])
                        span_labels.append(row[6])
                        start_token_to_orig[start_token] = ann_start_idx if row[4] == 1 else ann_start_idx - self.articles_df['added_chars_no'][idx]
                        end_token_to_orig[end_token] = ann_end_idx if row[4] == 1 else ann_end_idx - self.articles_df['added_chars_no'][idx]

                        # 1. Get the original text for comparison
                        original_entity = self.articles_df['text'][idx][ann_start_idx:ann_end_idx + 1]

                        # 2. Get the tokens from the input_ids
                        # input_ids is the tensor/list from the tokenizer encoding
                        token_ids = self.articles_df['input_ids'][idx][start_token : end_token + 1]
                        decoded_entity = tokenizer.decode(token_ids).strip()


                        # 3. Print and compare
                        if original_entity.lower() != decoded_entity.lower():
                            print(f"Mismatch found!")
                            print(f"Original: '{original_entity}'")
                            print(f"Decoded:  '{decoded_entity}'")
                            print(f"Indices:  {start_token}:{end_token}")
                            mismatches += 1

            self.dataset.append({
                "pmid":            pmid,
                "location":        self.annotations_df[self.annotations_df['pmid'] == pmid]['location'] if self.annotations_df is not None else None,
                "input_ids":       torch.tensor(self.articles_df.iloc[idx]['input_ids'], dtype=torch.long),
                "start_labels":    start_labels,
                "end_labels":      end_labels,
                "span_idxs":       span_idxs,
                "span_labels":     span_labels,
                "start_token_to_orig": start_token_to_orig,
                "end_token_to_orig": end_token_to_orig,
                "abstract_start_idx": self.articles_df['added_chars_no'][idx]
            })

    def __len__(self):
        return self.articles_df.shape[0]

    def __getitem__(self, idx):
        return self.dataset[idx]


In [ ]:
def collate_fn(batch):
    max_len = max(b["input_ids"].shape[0] for b in batch)
    B = len(batch)

    input_ids      = torch.zeros(B, max_len, dtype=torch.long)
    attention_mask = torch.zeros(B, max_len, dtype=torch.long)
    start_labels   = torch.zeros(B, max_len)
    end_labels     = torch.zeros(B, max_len)

    # match_labels[b, i, j] = 1  iff  (i, j) is a gold entity span in example b.
    # This is the supervision target for the bilinear span scorer (Tan et al. §3.3):
    # the scorer must learn to produce high scores for valid (start, end) pairs that
    # belong to the same entity, not just any independent start and end token.
    match_labels = torch.zeros(B, max_len, max_len)

    for i, b in enumerate(batch):
        L = b["input_ids"].shape[0]
        input_ids[i, :L]      = b["input_ids"]
        attention_mask[i, :L] = 1
        start_labels[i, :L]   = b["start_labels"]
        end_labels[i, :L]     = b["end_labels"]
        for (s, e) in b["span_idxs"]:
            match_labels[i, s, e] = 1.0   # mark every gold (start, end) pair

    span_idxs   = [b["span_idxs"]   for b in batch]
    span_labels = [b["span_labels"]  for b in batch]
    pmids       = [b["pmid"]         for b in batch]
    locations   = [b["location"]     for b in batch]

    return {
        "pmid":            pmids,
        "location":        locations,
        "input_ids":       input_ids,
        "attention_mask":  attention_mask,
        "start_labels":    start_labels,
        "end_labels":      end_labels,
        "match_labels":    match_labels,   # [B, max_len, max_len]
        "span_idxs":       span_idxs,
        "span_labels":     span_labels,
    }

## Model arhitecture

In [ ]:
@dataclass
class NERConfig:
    model_name: str = "thomas-sounack/BioClinical-ModernBERT-base"
    num_labels: int = 14          # 13 = non-entity, 0–12 = entity types
    max_seq_len: int = 512
    max_span_len: int = 20        # max token span to consider as candidate
    top_k_spans: int = 100        # top-k start-end pairs during inference

    # LoRA
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    lora_target_modules: List[str] = field(
        default_factory=lambda: ["query", "key", "value", "dense"]
    )

    # Training
    batch_size: int = 8
    grad_accum_steps: int = 4     # effective batch = 32
    lr_stage1: float = 2e-4
    lr_stage2: float = 2e-4
    lr_stage3: float = 1e-4
    warmup_ratio: float = 0.1
    epochs_stage1: int = 5
    epochs_stage2: int = 5
    epochs_stage3: int = 10

    # ── Boundary loss (Tan et al. AAAI 2020 §3.3) ──────────────────────────
    # L_boundary = lambda_s * L_start + lambda_e * L_end + lambda_m * L_match
    #
    # L_start / L_end: per-token binary cross-entropy
    #   label_i = 1 iff token i is the start (or end) of any gold entity span
    #
    # L_match: binary cross-entropy on the [L×L] bilinear score matrix
    #   match_label[i, j] = 1 iff (i, j) is an exact gold span (same entity)
    #   This directly trains the bilinear scorer to associate valid start-end pairs.
    #
    lambda_s: float = 1.0               # weight for start BCE loss
    lambda_e: float = 1.0               # weight for end BCE loss
    lambda_m: float = 0.5               # weight for bilinear matching BCE loss
    boundary_pos_weight: float = 10.0   # pos_weight for start/end BCE (sparsity)
    match_pos_weight: float = 20.0      # pos_weight for match BCE (even sparser)

    span_label_smoothing: float = 0.1

    # Hardware
    fp16: bool = True
    device: str = "cuda"

CFG = NERConfig()

In [ ]:
class BoundaryHead(nn.Module):
    """
    Three supervised components (Tan et al. AAAI 2020 §3.3):

      1. start_clf  — binary classifier: is token i the start of some entity?
      2. end_clf    — binary classifier: is token j the end of some entity?
      3. bilinear   — learned matching scorer: does (i, j) form a valid span?
                      Supervised by L_match: BCE against the gold (start, end) matrix.
    """

    def __init__(self, hidden_size: int):
        super().__init__()
        self.start_clf = nn.Linear(hidden_size, 1)
        self.end_clf   = nn.Linear(hidden_size, 1)
        # Bilinear: score(i, j) = h_i^T W h_j + b
        self.bilinear  = nn.Bilinear(hidden_size, hidden_size, 1, bias=True)

    def forward(
        self,
        hidden: torch.Tensor,   # [B, L, H]
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns:
            start_logits  [B, L]
            end_logits    [B, L]
            span_scores   [B, L, L]   — score(i, j) for all (start=i, end=j) pairs
        """
        start_logits = self.start_clf(hidden).squeeze(-1)   # [B, L]
        end_logits   = self.end_clf(hidden).squeeze(-1)     # [B, L]

        # Efficient bilinear over all pairs: reshape to [B*L*L, H] for one call
        B, L, H = hidden.shape
        h_s = hidden.unsqueeze(2).expand(B, L, L, H)  # [B, L, L, H]
        h_e = hidden.unsqueeze(1).expand(B, L, L, H)  # [B, L, L, H]
        span_scores = self.bilinear(
            h_s.reshape(B * L * L, H),
            h_e.reshape(B * L * L, H),
        ).reshape(B, L, L)                             # [B, L, L]

        return start_logits, end_logits, span_scores


In [ ]:
class SpanClassificationHead(nn.Module):
    """
    Mean-pools token representations for a span [s, e] and classifies it.
    Works on a variable list of (start, end) candidate spans per example.
    """

    def __init__(self, hidden_size: int, num_labels: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_labels),
        )

    def pool_span(
        self,
        hidden: torch.Tensor,    # [L, H]
        start: int,
        end: int,
    ) -> torch.Tensor:           # [H]
        return hidden[start : end + 1].mean(0)

    def forward(
        self,
        hidden: torch.Tensor,                    # [B, L, H]
        span_idxs: List[List[Tuple[int, int]]],  # per-example list of (s, e)
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            logits   [N_total_spans, num_labels]
            batch_id [N_total_spans]   — which example each span belongs to
        """
        all_logits, batch_ids = [], []
        for b_idx, spans in enumerate(span_idxs):
            if not spans:
                continue
            pooled = torch.stack(
                [self.pool_span(hidden[b_idx], s, e) for s, e in spans]
            )                          # [N_b, H]
            all_logits.append(self.mlp(pooled))
            batch_ids.extend([b_idx] * len(spans))

        if not all_logits:
            dummy = torch.zeros(0, self.mlp[-1].out_features, device=hidden.device)
            return dummy, []

        return torch.cat(all_logits, 0), batch_ids


In [ ]:
class DualHeadNER(nn.Module):
    """
    Full model: LoRA-wrapped encoder + boundary head + span classification head.
    """

    def __init__(self, cfg: NERConfig):
        super().__init__()
        self.cfg = cfg

        # Load base encoder and apply LoRA
        base_encoder = AutoModel.from_pretrained(cfg.model_name)
        lora_cfg = LoraConfig(
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            lora_dropout=cfg.lora_dropout,
            bias="none",
            target_modules=cfg.lora_target_modules,
        )
        self.encoder = get_peft_model(base_encoder, lora_cfg)

        H = self.encoder.config.hidden_size
        self.boundary_head = BoundaryHead(H)
        self.span_head     = SpanClassificationHead(H, cfg.num_labels)

    def encode(self, input_ids, attention_mask):
        return self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state      # [B, L, H]

    def forward(
        self,
        input_ids:       torch.Tensor,
        attention_mask:  torch.Tensor,
        span_idxs:       Optional[List] = None,   # None → skip span head
    ):
        hidden = self.encode(input_ids, attention_mask)
        start_logits, end_logits, span_scores = self.boundary_head(hidden)

        span_logits, batch_ids = None, None
        if span_idxs is not None:
            span_logits, batch_ids = self.span_head(hidden, span_idxs)

        return {
            "hidden":       hidden,
            "start_logits": start_logits,
            "end_logits":   end_logits,
            "span_scores":  span_scores,
            "span_logits":  span_logits,
            "batch_ids":    batch_ids,
        }

In [ ]:
def boundary_loss(
    start_logits:   torch.Tensor,   # [B, L]
    end_logits:     torch.Tensor,   # [B, L]
    span_scores:    torch.Tensor,   # [B, L, L]  — bilinear matching logits
    start_labels:   torch.Tensor,   # [B, L]
    end_labels:     torch.Tensor,   # [B, L]
    match_labels:   torch.Tensor,   # [B, L, L]  — 1 iff (i,j) is a gold span
    attention_mask: torch.Tensor,   # [B, L]
    cfg: NERConfig,
) -> torch.Tensor:
    """
    Three-component boundary loss from Tan et al. AAAI 2020 §3.3:

      L_boundary = lambda_s * L_start
                 + lambda_e * L_end
                 + lambda_m * L_match

    L_start, L_end:
        Per-token BCE, restricted to non-padding positions.
        pos_weight compensates for the rarity of entity boundary tokens.

    L_match:
        BCE on the full [L×L] bilinear score matrix.
        Only upper-triangular positions where i <= j are valid spans;
        padding rows/columns are masked out.
        match_labels[b, i, j] = 1 iff token i starts and token j ends
    """
    seq_mask = attention_mask.bool()   # [B, L]

    # ── L_start and L_end ────────────────────────────────────────────────────
    pw_se = torch.tensor(cfg.boundary_pos_weight, device=start_logits.device)
    bce_se = nn.BCEWithLogitsLoss(pos_weight=pw_se, reduction="none")

    loss_s = bce_se(start_logits, start_labels)[seq_mask].mean()
    loss_e = bce_se(end_logits,   end_labels)  [seq_mask].mean()

    # ── L_match ──────────────────────────────────────────────────────────────
    # Valid positions: i <= j (upper triangle) AND both i, j are non-padding.
    # Build a [B, L, L] boolean mask: True where we want to compute loss.
    L = attention_mask.shape[1]
    # upper-triangular mask (i <= j), same for every batch element
    upper_tri = torch.triu(
        torch.ones(L, L, dtype=torch.bool, device=span_scores.device)
    )                                              # [L, L]
    # row mask: token i must not be padding
    row_mask = seq_mask.unsqueeze(2)               # [B, L, 1]
    # col mask: token j must not be padding
    col_mask = seq_mask.unsqueeze(1)               # [B, 1, L]
    pair_mask = upper_tri & row_mask & col_mask    # [B, L, L]

    pw_m = torch.tensor(cfg.match_pos_weight, device=span_scores.device)
    bce_m = nn.BCEWithLogitsLoss(pos_weight=pw_m, reduction="none")

    loss_m = bce_m(span_scores, match_labels)[pair_mask].mean()

    return cfg.lambda_s * loss_s + cfg.lambda_e * loss_e + cfg.lambda_m * loss_m



def span_loss(
    span_logits: torch.Tensor,    # [N, num_labels]
    span_labels: List[List[int]], # per-example lists
    label_smoothing: float,
) -> torch.Tensor:
    flat_labels = torch.tensor(
        [lbl for lbls in span_labels for lbl in lbls],
        dtype=torch.long,
        device=span_logits.device,
    )
    if flat_labels.numel() == 0:
        return span_logits.sum() * 0.0
    return F.cross_entropy(
        span_logits, flat_labels,
        label_smoothing=label_smoothing,
    )

## Training loop

In [ ]:
def freeze(module: nn.Module):
    for p in module.parameters():
        p.requires_grad_(False)

def unfreeze(module: nn.Module):
    for p in module.parameters():
        p.requires_grad_(True)

def set_stage(model: DualHeadNER, stage: int):
    """
    Stage 1: boundary head + LoRA active; span head frozen
    Stage 2: span head + LoRA active; boundary head frozen
    Stage 3: everything active
    """
    if stage == 1:
        unfreeze(model.encoder)
        unfreeze(model.boundary_head)
        freeze(model.span_head)

    elif stage == 2:
        freeze(model.boundary_head)
        unfreeze(model.encoder)
        unfreeze(model.span_head)

    elif stage == 3:
        unfreeze(model.encoder)
        unfreeze(model.boundary_head)
        unfreeze(model.span_head)

    # LoRA base weights stay frozen in all stages
    for name, param in model.encoder.named_parameters():
        if "lora_" not in name:
            param.requires_grad_(False)

In [ ]:
@torch.no_grad()
def predict(
    model:     DualHeadNER,
    batch:     dict,
    cfg:       NERConfig,
    threshold: float = 0.5,
) -> List[List[Tuple[int, int, int]]]:
    """
    Returns per-example list of (start, end, label) predictions.

    Pipeline:
      1. Boundary head → start_probs, end_probs, bilinear span_scores
      2. Combined score = start_prob(i) * end_prob(j) * sigmoid(span_score(i,j))
      3. Top-k candidates above threshold → span head → entity labels
    """
    device = next(model.parameters()).device
    ids  = batch["input_ids"].to(device)
    mask = batch["attention_mask"].to(device)

    out = model(ids, mask, span_idxs=None)   # boundary only
    start_probs = torch.sigmoid(out["start_logits"])  # [B, L]
    end_probs   = torch.sigmoid(out["end_logits"])    # [B, L]
    span_scores = out["span_scores"]                  # [B, L, L]
    hidden      = out["hidden"]

    B, L = ids.shape
    results = []

    for b in range(B):
        seq_len = mask[b].sum().item()
        s_p = start_probs[b, :seq_len]
        e_p = end_probs[b, :seq_len]
        sc  = span_scores[b, :seq_len, :seq_len]  # [seq_len, seq_len]

        # Valid candidate mask: i <= j, span length <= max_span_len
        cand_mask = torch.tril(
            torch.ones(seq_len, seq_len, device=device, dtype=torch.bool),
            diagonal=cfg.max_span_len,
        ) & torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool))

        combined = s_p.unsqueeze(1) * e_p.unsqueeze(0) * torch.sigmoid(sc)
        combined = combined * cand_mask.float()

        # Top-k
        flat = combined.view(-1)
        k = min(cfg.top_k_spans, (flat > 0).sum().item())
        if k == 0:
            results.append([])
            continue

        top_vals, top_idx = flat.topk(k)
        starts = (top_idx // seq_len).tolist()
        ends   = (top_idx %  seq_len).tolist()
        cands  = [(s, e) for s, e, v in zip(starts, ends, top_vals.tolist())
                  if v > threshold]

        if not cands:
            results.append([])
            continue

        # Classify candidates with span head
        span_logits, _ = model.span_head(hidden[b:b+1], [cands])
        pred_labels = span_logits.argmax(-1).tolist()

        preds = []
        for (s, e), lbl in zip(cands, pred_labels):
            if lbl != 0:
                preds.append((s, e, lbl))
        results.append(preds)

    return results

In [ ]:
@torch.no_grad()
def evaluate(
    model:  DualHeadNER,
    loader: DataLoader,
    cfg:    NERConfig,
) -> dict:
    model.eval()
    device = next(model.parameters()).device

    # Track stats per class: {label_id: {"tp": 0, "fp": 0, "fn": 0}}
    class_stats = {i: {"tp": 0, "fp": 0, "fn": 0} for i in range(1, 14)}

    for batch in loader:
        preds_batch = predict(model, batch, cfg)

        for b_idx, preds in enumerate(preds_batch):
            gold = set(
                (s, e, lbl)
                for (s, e), lbl in zip(
                    batch["span_idxs"][b_idx],
                    batch["span_labels"][b_idx],
                )
                if lbl != 0
            )
            pred_set = set(preds)

            # True Positives
            for span in (pred_set & gold):
                class_stats[span[2]]["tp"] += 1

            # False Positives (predicted but not in gold)
            for span in (pred_set - gold):
                class_stats[span[2]]["fp"] += 1

            # False Negatives (gold but not predicted)
            for span in (gold - pred_set):
                class_stats[span[2]]["fn"] += 1

    # --- Metrics Calculation ---
    total_tp = total_fp = total_fn = 0
    macro_prec = macro_rec = 0

    for lbl, stats in class_stats.items():
        tp, fp, fn = stats["tp"], stats["fp"], stats["fn"]
        total_tp += tp
        total_fp += fp
        total_fn += fn

        # Per-class Precision/Recall for Macro
        p = tp / (tp + fp + 1e-9)
        r = tp / (tp + fn + 1e-9)
        macro_prec += p
        macro_rec += r

    # Micro-Average
    micro_prec = total_tp / (total_tp + total_fp + 1e-9)
    micro_rec  = total_tp / (total_tp + total_fn + 1e-9)
    micro_f1   = 2 * micro_prec * micro_rec / (micro_prec + micro_rec + 1e-9)

    # Macro-Average
    macro_prec /= 13
    macro_rec /= 13
    macro_f1 = 2 * macro_prec * macro_rec / (macro_prec + macro_rec + 1e-9)

    return {
        "micro_f1": micro_f1,
        "micro_precision": micro_prec,
        "micro_recall": micro_rec,
        "macro_f1": macro_f1,
        "class_stats": class_stats
    }


In [ ]:
def save_checkpoint(state, stage, epoch, path="checkpoint.pth"):
    # Save to a temporary file
    torch.save(state, f"stage_{stage}_epoch_{epoch}_{path}")
    # Maintain a 'latest' symlink or copy for easy resumption
    torch.save(state, f"latest_stage_{stage}.pth")
    print(f"  --> Checkpoint saved: stage_{stage}_epoch_{epoch}")

In [ ]:
def make_optimizer(model: DualHeadNER, lr: float):
    trainable = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-2)


def train_stage(
    model:      DualHeadNER,
    loader:     DataLoader,
    stage:      int,
    epochs:     int,
    lr:         float,
    cfg:        NERConfig,
    val_loader: Optional[DataLoader] = None,
    scaler:     Optional[torch.cuda.amp.GradScaler] = None,
):
    set_stage(model, stage)
    optimizer = make_optimizer(model, lr)
    total_steps = (len(loader) // cfg.grad_accum_steps) * epochs
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    device = torch.device(cfg.device)
    model.to(device)
    use_fp16 = cfg.fp16 and device.type == "cuda"
    if use_fp16 and scaler is None:
        scaler = torch.cuda.amp.GradScaler()

    print(f"\n{'='*55}")
    print(f"Stage {stage} | {epochs} epochs | lr={lr}")
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable_params:,}")
    print('='*55)

    model.train()
    optimizer.zero_grad()

    for epoch in range(epochs):
        epoch_loss, n_steps = 0.0, 0

        for step, batch in enumerate(loader):
            ids      = batch["input_ids"].to(device)
            mask     = batch["attention_mask"].to(device)
            s_lbl    = batch["start_labels"].to(device)
            e_lbl    = batch["end_labels"].to(device)
            m_lbl    = batch["match_labels"].to(device)   # [B, L, L]

            # Only pass span_idxs when the span head is active
            span_idxs   = batch["span_idxs"]   if stage >= 2 else None
            span_labels = batch["span_labels"]  if stage >= 2 else None

            with torch.cuda.amp.autocast(enabled=use_fp16):
                out = model(ids, mask, span_idxs=span_idxs)

                loss = torch.tensor(0.0, device=device)

                if stage in (1, 3):
                    # All three boundary losses: L_start, L_end, L_match.
                    loss = loss + boundary_loss(
                        out["start_logits"], out["end_logits"], out["span_scores"],
                        s_lbl, e_lbl, m_lbl, mask,
                        cfg,
                    )

                if stage in (2, 3) and out["span_logits"] is not None:
                    loss = loss + span_loss(
                        out["span_logits"],
                        span_labels,
                        cfg.span_label_smoothing,
                    )

                loss = loss / cfg.grad_accum_steps

            if use_fp16:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (step + 1) % cfg.grad_accum_steps == 0:
                if use_fp16:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                n_steps += 1

            epoch_loss += loss.item() * cfg.grad_accum_steps

        avg = epoch_loss / len(loader)
        print(f"  Epoch {epoch+1}/{epochs} | loss={avg:.4f}")

        if val_loader is not None:
            val = evaluate(model, val_loader, cfg)
            print(f"  Val: {val}")
            model.train()

        checkpoint_state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'stage': stage,
        }
        if use_fp16:
            checkpoint_state['scaler_state_dict'] = scaler.state_dict()

        save_checkpoint(checkpoint_state, stage, epoch)

    return scaler

In [ ]:
def train_pipeline(
    train_bronze_ds: GBDataset,
    train_silver_gold_ds: GBDataset,
    val_ds:   GBDataset,
    cfg:      NERConfig = CFG,
) -> DualHeadNER:
    """
    End-to-end: initialise model, run stages 1–3, return trained model.
    """
    model = DualHeadNER(cfg)

    train_bronze_loader = DataLoader(
        train_bronze_ds, batch_size=cfg.batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )
    train_silver_loader = DataLoader(
        train_silver_gold_ds, batch_size=cfg.batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )

    scaler = torch.cuda.amp.GradScaler() if cfg.fp16 else None

    # Stage 1 — boundary head
    scaler = train_stage(
        model, train_bronze_loader, stage=1,
        epochs=cfg.epochs_stage1, lr=cfg.lr_stage1,
        cfg=cfg, val_loader=val_loader, scaler=scaler,
    )

    # Stage 2 — span classification head (boundary frozen)
    scaler = train_stage(
        model, train_bronze_loader, stage=2,
        epochs=cfg.epochs_stage2, lr=cfg.lr_stage2,
        cfg=cfg, val_loader=val_loader, scaler=scaler,
    )

    # Stage 3 — joint fine-tuning
    scaler = train_stage(
        model, train_silver_gold_ds, stage=3,
        epochs=cfg.epochs_stage3, lr=cfg.lr_stage3,
        cfg=cfg, val_loader=val_loader, scaler=scaler,
    )

    return model

## Predictions

In [ ]:
def format_predictions(
    model:     DualHeadNER,
    loader:    DataLoader,
    tokenizer: AutoTokenizer,
    cfg:       NERConfig,
) -> pd.DataFrame:

    model.eval()
    results = {}

    for batch in loader:
        preds_batch = predict(model, batch, cfg)

        for b_idx, preds in enumerate(preds_batch):
            pmid     = batch["pmid"][b_idx]
            ids      = batch["input_ids"][b_idx].tolist()
            start_token_to_orig = batch["start_token_to_orig"][b_idx]
            end_token_to_orig = batch["end_token_to_orig"][b_idx]
            abstract_start_idx = batch["abstract_start_idx"][b_idx]
            results[pmid] = {
                "entities": []
            }

            for (s, e, lbl) in preds:
                span_ids = ids[s : e + 1]
                text     = tokenizer.decode(span_ids, skip_special_tokens=True)
                text = text.removesuffix(" ,.;:-()[]")
                text = text.removeprefix(" ,.;:-()[]")

                results[pmid]["entities"].append({
                    "start_idx": start_token_to_orig[s],
                    "end_idx":   end_token_to_orig[e],
                    "location": "abstract" if start_token_to_orig[s] >= abstract_start_idx else "title",
                    "text_span":   text,
                    "label":       lbl
                })

    return results

## Run

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
train_bronze_ds = torch.load("./datasets/bronze_dataset.pt")
train_silver_ds = torch.load("./datasets/silver_dataset.pt")
train_silver_2025_ds = torch.load("./datasets/silver2025_dataset.pt")
train_gold_ds = torch.load("./datasets/gold_dataset.pt")
train_silver_gold_ds = ConcatDataset([train_silver_ds, train_silver_2025_ds, train_gold_ds])
val_ds = torch.load("./datasets/dev_dataset.pt")

In [ ]:
test_ds     = torch.load("./datasets/test_dataset.pt")
test_loader = DataLoader(test_ds, batch_size=16, collate_fn=collate_fn)

In [ ]:
model = train_pipeline(train_bronze_ds, train_silver_gold_ds, val_ds, CFG)
torch.save(model.state_dict(), "gutbrainie_model.pt")

In [ ]:
submission  = format_predictions(model, test_loader, tokenizer, CFG)
submission.to_csv("submission.csv", index=False)

## References

* Sounack, Thomas, et al. "Bioclinical modernbert: A state-of-the-art long-context encoder for biomedical and clinical nlp." arXiv preprint arXiv:2506.10896 (2025).
* Li, Xiaoya, et al. "A unified MRC framework for named entity recognition." Proceedings of the 58th annual meeting of the association for computational linguistics. 2020.
* Gemini, Claude, Chat GPT